In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small/checkpoints/post_cell_20_try_3.pickle

In [ ]:
%%cudf.pandas.profile
### cell 21 ###

### cell 21 optimized ###

# define the question and (possibly blank) x-axis title
question_of_interest_cell33 = 'For how many years have you been writing code and/or programming?'
title_for_x_axis_cell33 = ''

# combine percentages for multiple years into one DataFrame using vectorized GPU operations
def combine_subset_of_data_from_multiple_years_33(question_of_interest, x_axis_title, include_2017=False):
    year_sources = [
        ('2018', responses_df_2018_cell10, 'How long have you been writing code to analyze data?'),
        ('2019', responses_df_2019_cell10, 'How long have you been writing code to analyze data (at work or at school)?'),
        ('2020', responses_df_2020, None),
        ('2021', responses_df_2021, None),
        ('2022', responses_df_2022_cell10, None),
    ]
    if include_2017:
        year_sources.insert(0, ('2017', responses_df_2017, None))

    # year-specific recoding maps
    replace_maps = {
        '2018': {
            '< 1 year': '< 1 years',
            'I have never written code but I want to learn': 'I have never written code',
            'I have never written code and I do not want to learn': 'I have never written code',
            '1-2 years': '1-3 years',
            '20-30 years': '20+ years',
            '30-40 years': '20+ years',
            '40+ years': '20+ years',
        },
        '2019': {'1-2 years': '1-3 years'},
        '2020': {'1-2 years': '1-3 years'},
    }
    # build a GPU DataFrame of mapping entries
    map_entries = []
    for yr, mp in replace_maps.items():
        for orig, new in mp.items():
            map_entries.append({'year': yr, 'orig': orig, 'new': new})
    map_df = pd.DataFrame(map_entries)

    # concatenate all responses with a year column
    dfs = []
    for year, df_src, orig_col in year_sources:
        # pick or rename the column to the canonical question
        if orig_col:
            s = df_src[orig_col]
            s.name = question_of_interest
            df_tmp = s.to_frame()
        else:
            df_tmp = df_src[[question_of_interest]]
        df_tmp['year'] = year
        dfs.append(df_tmp)
    df_combined = pd.concat(dfs, ignore_index=True)

    # apply year-specific recoding via a GPU merge
    df_combined = df_combined.merge(map_df, how='left', on=['year'])
    df_combined[question_of_interest] = df_combined['new'].fillna(df_combined[question_of_interest])
    df_combined = df_combined.drop(columns=['orig', 'new'])

    # compute counts and percentages with GPU groupby
    df_counts = (
        df_combined.groupby(['year', question_of_interest])
        .size()
        .reset_index(name='count')
    )
    df_counts['percentage'] = (
        (df_counts['count'] * 100 /
         df_counts.groupby('year')['count'].transform('sum'))
        .round(1)
    )

    # rename category column if needed
    if x_axis_title:
        df_counts = df_counts.rename(columns={question_of_interest: x_axis_title})
    else:
        df_counts = df_counts.rename(columns={question_of_interest: ''})

    return df_counts[[x_axis_title or '', 'percentage', 'year']]

# run and inspect
programming_df_combined = (
    combine_subset_of_data_from_multiple_years_33(
        question_of_interest_cell33,
        title_for_x_axis_cell33
    )
    .sort_values(['year', 'percentage'])
)
programming_df_combined.info()